In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_circuits_transformed = spark.table(f"{catalogo}.{esquema_source}.circuits_transformed")

In [0]:
df_transformed = df_circuits_transformed.groupBy(col("race_year")).agg(
                                                     count(col("location")).alias("conteo"),
                                                     max(col("altitude")).alias("max_altitude"),
                                                     min(col("altitude")).alias("min_altitude"),
                                                     max(col("country")).alias("country"),
                                                     max(col("race_type")).alias("race_type"),
                                                     max(col("near_equator")).alias("near_equator")
                                                     ).orderBy(col("race_year").desc())

In [0]:
df_transformed.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.golden_raced_partitioned")

In [0]:
%sql

DESCRIBE TABLE catalog_au.golden.dim_tiempo;

In [0]:
%sql
SHOW CREATE TABLE catalog_au.golden.dim_tiempo;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT sk_tiempo) AS claves_distintas,
    MIN(fecha) AS primera_fecha,
    MAX(fecha) AS ultima_fecha
FROM catalog_au.golden.dim_tiempo;

In [0]:

from pyspark.sql import functions as F

# 1. Rango de fechas
fecha_inicio = "2024-01-01"
fecha_fin = "2026-12-31"

# 2. Generar una fila por día
df_dim_tiempo = spark.sql(f"""
    SELECT explode(
        sequence(
            to_date('{fecha_inicio}'),
            to_date('{fecha_fin}'),
            interval 1 day
        )
    ) AS fecha
""")

# 3. Nombres en español
meses = [
    "Enero", "Febrero", "Marzo", "Abril",
    "Mayo", "Junio", "Julio", "Agosto",
    "Septiembre", "Octubre", "Noviembre", "Diciembre"
]

dias = [
    "Lunes", "Martes", "Miércoles",
    "Jueves", "Viernes", "Sábado", "Domingo"
]

# 4. Construir los atributos de la dimensión
df_dim_tiempo = (
    df_dim_tiempo
    .withColumn("sk_tiempo", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("semestre", F.when(F.month("fecha") <= 6, 1).otherwise(2))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn(
        "nombre_mes",
        F.element_at(
            F.array(*[F.lit(m) for m in meses]),
            F.col("mes")
        )
    )
    .withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("dia_semana", F.weekday("fecha") + 1)
    .withColumn(
        "nombre_dia",
        F.element_at(
            F.array(*[F.lit(d) for d in dias]),
            F.col("dia_semana")
        )
    )
    .withColumn("es_fin_mes", F.col("fecha") == F.last_day("fecha"))
    .withColumn(
        "es_fin_trimestre",
        (F.col("mes") % 3 == 0) & F.col("es_fin_mes")
    )
    .withColumn(
        "es_fin_anio",
        (F.col("mes") == 12) & F.col("es_fin_mes")
    )
    .withColumn("fecha_creacion", F.current_timestamp())
    .select(
        "sk_tiempo", "fecha", "anio", "semestre",
        "trimestre", "mes", "nombre_mes",
        "dia", "dia_semana", "nombre_dia",
        "es_fin_mes", "es_fin_trimestre",
        "es_fin_anio", "fecha_creacion"
    )
)

print("Registros generados:", df_dim_tiempo.count())
display(df_dim_tiempo.orderBy("fecha").limit(10))


In [0]:
from pyspark.sql import functions as F

display(
    df_dim_tiempo.agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("sk_tiempo").alias("claves_distintas"),
        F.countDistinct("fecha").alias("fechas_distintas"),
        F.min("fecha").alias("primera_fecha"),
        F.max("fecha").alias("ultima_fecha"),
        F.sum(
            F.when(
                F.col("sk_tiempo").isNull() |
                F.col("fecha").isNull(), 1
            ).otherwise(0)
        ).alias("registros_nulos")
    )
)

In [0]:
display(
    df_dim_tiempo.agg(
        F.sum(
            F.col("es_fin_mes").cast("int")
        ).alias("cierres_mes"),

        F.sum(
            F.col("es_fin_trimestre").cast("int")
        ).alias("cierres_trimestre"),

        F.sum(
            F.col("es_fin_anio").cast("int")
        ).alias("cierres_anio"),

        F.sum(
            F.when(
                F.col("fecha") == F.lit("2024-02-29").cast("date"),
                1
            ).otherwise(0)
        ).alias("dia_bisiesto_2024")
    )
)

In [0]:
total = df_dim_tiempo.count()
claves = df_dim_tiempo.select("sk_tiempo").distinct().count()
fechas = df_dim_tiempo.select("fecha").distinct().count()

nulos = df_dim_tiempo.filter(
    F.col("sk_tiempo").isNull() |
    F.col("fecha").isNull()
).count()

assert total == 1096, f"Total incorrecto: {total}"
assert claves == total, "Existen claves duplicadas"
assert fechas == total, "Existen fechas duplicadas"
assert nulos == 0, "Existen claves o fechas nulas"

print("VALIDACIÓN EXITOSA")
print(f"Registros: {total}")
print(f"Claves únicas: {claves}")
print(f"Fechas únicas: {fechas}")
print(f"Registros nulos: {nulos}")

In [0]:
df_dim_tiempo.createOrReplaceTempView(
    "vw_dim_tiempo_carga"
)

print("Vista temporal creada correctamente.")
print("Registros preparados:", df_dim_tiempo.count())

In [0]:
%sql

MERGE INTO catalog_au.golden.dim_tiempo AS destino

USING vw_dim_tiempo_carga AS origen

ON destino.sk_tiempo = origen.sk_tiempo

WHEN NOT MATCHED THEN
INSERT (
    sk_tiempo,
    fecha,
    anio,
    semestre,
    trimestre,
    mes,
    nombre_mes,
    dia,
    dia_semana,
    nombre_dia,
    es_fin_mes,
    es_fin_trimestre,
    es_fin_anio,
    fecha_creacion
)
VALUES (
    origen.sk_tiempo,
    origen.fecha,
    origen.anio,
    origen.semestre,
    origen.trimestre,
    origen.mes,
    origen.nombre_mes,
    origen.dia,
    origen.dia_semana,
    origen.nombre_dia,
    origen.es_fin_mes,
    origen.es_fin_trimestre,
    origen.es_fin_anio,
    origen.fecha_creacion
);

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT sk_tiempo) AS claves_distintas,
    COUNT(DISTINCT fecha) AS fechas_distintas,
    MIN(fecha) AS primera_fecha,
    MAX(fecha) AS ultima_fecha
FROM catalog_au.golden.dim_tiempo;

In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM catalog_au.golden.dim_concepto_financiero;